## 다층 퍼셉트론(Multilayer perceptron, MLP) 최적화 기법

In [7]:
import numpy as np
# 행렬 연산 라이브러리 불러오기

#### 행렬 연산으로 구현한 MLP (Mini batch)

- 7장에서는 훈련 샘플 **전체(batch)를 행렬로 묶어서 한 번에** 순방향/역방향 계산
- 샘플별로 계산된 기울기가 행렬 곱에서 **합산**되므로, **batch size(여기서는 4)로 나누어 평균**을 낸 후 가중치를 갱신
- 입력 2개 → 은닉 2개 → 출력 1개, 활성화 함수는 모두 시그모이드 함수
- 행렬 크기: $X$ (4×2), $W_1$ (2×2), $W_2$ (2×1), $B_1$ (2,), $B_2$ (1,)

In [8]:
class ActivationFunctions:
    """입력의 총합을 받아서 출력값을 계산하는 활성화 함수 클래스"""

    def __init__(self, value):
        self.value = value
        # 활성화 함수에 넣을 입력값 (스칼라 / 넘파이 배열 모두 가능)

    def sigmoidFunction(self):
        """시그모이드 함수 메서드
        Logic:
            - 실수 입력값을 0과 1 사이의 값으로 변환, 전 구간 미분 가능
            - 1 / (1 + e**(-x))
        """
        return 1.0 / (1.0 + np.exp(-self.value))
        # 입력 총합을 0~1 사이 값으로 변환

    def sigmoidDerivative(self):
        """시그모이드 함수의 미분 메서드
        Logic:
            - f'(z) = f(z) * (1 - f(z))
            - 주의 : value 는 z가 아니라 '시그모이드를 통과한 출력값 f(z)' 이어야 함
        """
        return self.value * (1.0 - self.value)

In [ ]:
class MultiLayerPerceptron:
    """행렬 연산으로 구현한 다층 퍼셉트론 (입력층 - 은닉층 - 출력층) 클래스"""

    def __init__(self, inputs=2, hiddens=2, outputs=1, learningRate=0.5, epochs=10000, seed=0):
        """신경망 구조와 하이퍼파라미터를 설정하고 가중치를 초기화"""
        self.learningRate = learningRate
        # 학습률 : 가중치를 한 번에 얼마나 변경할지 결정
        self.epochs = epochs
        # 에폭 : 전체 데이터(batch)로 가중치 갱신을 몇 번 반복할지 횟수
        self.lossHistory = []
        # 학습 중 손실값을 기록하는 리스트 (학습이 진행되는지 확인용)

        np.random.seed(seed)
        # 난수 시드 고정 : 실행할 때마다 같은 결과가 나오도록 함

        self.W1 = 2 * np.random.random((inputs, hiddens)) - 1
        # 입력층 → 은닉층 가중치 : (inputs, hiddens), 0~1 난수를 2배 후 1을 빼서 -1~1 범위로 변환
        self.W2 = 2 * np.random.random((hiddens, outputs)) - 1
        # 은닉층 → 출력층 가중치 : (hiddens, outputs)
        self.B1 = np.zeros(hiddens)
        # 은닉층 바이어스 : (hiddens,), 0으로 초기화
        self.B2 = np.zeros(outputs)
        # 출력층 바이어스 : (outputs,), 0으로 초기화

    def predict(self, x):
        """순방향 전파 메서드
        Logic:
            - 입력 → (행렬 곱 + 바이어스) → 활성화 함수 → 은닉층 출력
            - 은닉층 출력 → (행렬 곱 + 바이어스) → 활성화 함수 → 출력층 출력
        """
        layer0 = x
        # 입력을 layer0에 대입
        Z1 = np.dot(layer0, self.W1) + self.B1
        # 행렬의 곱을 계산 (N, inputs) x (inputs, hiddens) → (N, hiddens)

        layer1 = ActivationFunctions(Z1).sigmoidFunction()
        # 활성화 함수를 적용 (은닉층 출력)
        Z2 = np.dot(layer1, self.W2) + self.B2
        # 행렬의 곱을 계산 (N, hiddens) x (hiddens, outputs) → (N, outputs)

        layer2 = ActivationFunctions(Z2).sigmoidFunction()
        # 활성화 함수를 적용 (출력층 출력)

        return layer0, layer1, layer2
        # 역전파에서 쓰기 위해 각 층의 출력을 모두 반환

    def fit(self, X, T):
        """역전파 학습 메서드
        Logic:
            - batch 전체를 행렬로 만들어서 한 번에 순방향/역방향 계산
            - 델타를 역방향으로 전파하고, 가중치는 batch size 로 나눈 평균 기울기로 갱신
        """
        batchSize = X.shape[0]
        # batch 크기 : 샘플 개수 (XOR 은 4)

        for i in range(self.epochs):
            layer0, layer1, layer2 = self.predict(X)
            # 순방향 계산 : batch 전체를 행렬로 만들어 한 번에 처리

            layer2_error = layer2 - T
            # 출력층 오차 계산 : 예측값 - 정답, (N, outputs)

            layer2_delta = layer2_error * ActivationFunctions(layer2).sigmoidDerivative()
            # 출력층 델타 계산 : 같은 위치끼리 곱하는 아다마르 곱(*)

            layer1_error = np.dot(layer2_delta, self.W2.T)
            # 은닉층 오차 계산 : 출력층 델타를 W2의 전치 행렬로 역전파, (N, outputs) x (outputs, hiddens)

            layer1_delta = layer1_error * ActivationFunctions(layer1).sigmoidDerivative()
            # 은닉층 델타 계산 : 아다마르 곱

            self.W2 += -self.learningRate * np.dot(layer1.T, layer2_delta) / batchSize
            # 은닉→출력 가중치 갱신 : (hiddens, N) x (N, outputs), batch size 로 나눠 평균 기울기 사용
            self.W1 += -self.learningRate * np.dot(layer0.T, layer1_delta) / batchSize
            # 입력→은닉 가중치 갱신 : (inputs, N) x (N, hiddens)
            self.B2 += -self.learningRate * np.sum(layer2_delta, axis=0) / batchSize
            # 출력층 바이어스 갱신 : 델타의 열 합계 (바이어스의 입력은 1로 가정)
            self.B1 += -self.learningRate * np.sum(layer1_delta, axis=0) / batchSize
            # 은닉층 바이어스 갱신

            loss = 0.5 * np.sum(layer2_error ** 2)
            # 손실함수 : 1/2 * 오차 제곱합
            self.lossHistory.append(loss)
            # 손실값 기록
            if i % 2000 == 0:
                print(f"epoch {i:5d}, loss = {loss:.6f}")
                # 2000번마다 손실값 출력

    def test(self, X, T):
        """학습 결과 확인 메서드
        Logic:        
        - 입력, 정답, 신경망 출력을 샘플별로 출력"""
        for x, y in zip(X, T):
            # 훈련 샘플과 정답을 하나씩 반복
            x = np.reshape(x, (1, -1))
            # 하나여도 2차원 형태 [0, 1] → [[0, 1]]
            layer0, layer1, layer2 = self.predict(x)
            # 학습된 가중치로 순방향 계산
            print(x, y, layer2)
            # 입력, 정답, 신경망 출력 비교

| 구분 | 6장 (온라인 학습) | 7장 (행렬 batch 학습) |
|---|---|---|
| 처리 단위 | 샘플 1개씩 | 샘플 4개를 행렬로 묶어서 |
| 가중치 갱신 | 샘플마다 | 에폭마다 1번 (평균 기울기) |
| 기울기 처리 | 그대로 사용 | batch size(4)로 나눔 |
| 코드 | `for x, y in zip(X, T)` 안에서 `reshape` | `predict(X)` 한 번으로 전체 처리 |

In [11]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
# 훈련 입력 (XOR) : (4, 2) 행렬, 행 = 샘플, 열 = 특징
T = np.array([[0], [1], [1], [0]])
# 정답 : (4, 1) 행렬

mlp = MultiLayerPerceptron(
    inputs = 2, hiddens = 2, outputs = 1, 
    learningRate = 0.5, epochs = 10000)
# 2 → 2 → 1 구조의 신경망 객체 생성

mlp.fit(X, T)
# 학습 실행 : 10000번 반복하면서 가중치 갱신

mlp.test(X, T)
# 학습 결과 확인 : 출력이 정답(0 또는 1)에 가까워야 한다.

epoch     0, loss = 0.500835
epoch  2000, loss = 0.499770
epoch  4000, loss = 0.486523
epoch  6000, loss = 0.149574
epoch  8000, loss = 0.015695
[[0 0]] [0] [[0.06189842]]
[[0 1]] [1] [[0.94296687]]
[[1 0]] [1] [[0.9429466]]
[[1 1]] [0] [[0.06157222]]
